STEP 1 — Prepare Input File (URL List)
links_for_golbal_military_data.txt


STEP 2 — Import Required Libraries

In [4]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re


Step 3 : Final code


In [ ]:
"""
Project: Unified Military Analytics & Comparison Dashboard

Description:
This script extracts military-related data from GlobalFirepower.com.
It collects country-wise military metrics such as manpower, manpower
availability, and related indicators, and merges them into a structured
CSV file for further analysis, visualization, and dashboard creation.


Author: Rutuja Ghodake  Date: 1st january 2026
"""

# -------------------------------
# Import Required Libraries
# -------------------------------
import requests                    # To send HTTP requests
from bs4 import BeautifulSoup      # To parse HTML content
import pandas as pd                # For data manipulation
import re                          # For cleaning numeric values


# -------------------------------
# Global Headers (Avoid blocking)
# -------------------------------
HEADERS = {
    "User-Agent": "Mozilla/5.0"
}


# -------------------------------------------------------
# Function: read_links_txt
# Purpose : Read URLs from a text file and clean them
# -------------------------------------------------------
def read_links_txt(path):
    """
    Reads a text file containing URLs and returns a clean list.

    Args:
        path (str): Path to the text file containing URLs.

    Returns:
        list: List of valid URLs.
    """
    links = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            # Skip empty lines
            if not line:
                continue

            # Only keep valid URLs
            if line.startswith("http"):
                links.append(line)

    return links


# -------------------------------------------------------
# Main Scraping Function
# -------------------------------------------------------
def scrape_global_firepower():
    """
    Scrapes military data from GlobalFirepower website.
    Combines base country ranking with multiple metric pages.
    Saves final output as a CSV file.
    """

    # Base ranking page (contains country list)
    base_url = "https://www.globalfirepower.com/countries-listing.php"

    # File containing metric URLs
    metric_urls = read_links_txt("links_for_golbal_military_data.txt")

    # --------------------------------------------------
    # STEP 1: Scrape country names & ranks
    # --------------------------------------------------
    response = requests.get(base_url, headers=HEADERS)
    soup = BeautifulSoup(response.text, "html.parser")

    containers = soup.select("div.picTrans.recordsetContainer")

    countries = []
    ranks = []

    for box in containers:
        try:
            country = box.find("span", class_="textWhite textLarge textShadow").text.strip()
            rank = box.find("span", class_="textWhite textLarge textBold").text.strip()
            countries.append(country)
            ranks.append(rank)
        except:
            continue  # Skip broken rows safely

    # Base dataframe
    df = pd.DataFrame({
        "Country": countries,
        "Rank": ranks
    })

    # --------------------------------------------------
    # STEP 2: Loop through all metric URLs
    # --------------------------------------------------
    for url in metric_urls:
        print(f"Scraping: {url}")

        page = requests.get(url, headers=HEADERS)
        soup = BeautifulSoup(page.text, "html.parser")

        rows = soup.select("div.picTrans.recordsetContainer")

        country_list = []
        metric_values = []

        for row in rows:
            try:
                country = row.find("span", class_="textWhite textLarge textShadow").text.strip()
                value = row.find_all("span", class_="textWhite textLarge")[-1].text.strip()
                country_list.append(country)
                metric_values.append(value)
            except:
                continue

        # Column name derived from URL
        column_name = url.split("/")[-1].replace(".php", "")

        temp_df = pd.DataFrame({
            "Country": country_list,
            column_name: metric_values
        })

        # Merge with main dataset
        df = df.merge(temp_df, on="Country", how="left")
        # Preview the first few rows after each merge
        print(df.head())

    # --------------------------------------------------
    # STEP 3: Save Final Output
    # --------------------------------------------------
    df.to_csv("Global_military_data_final.csv", index=False)


    print("✅ Data extraction completed successfully!")


# --------------------------------------------------
# Execute Script
# --------------------------------------------------
scrape_global_firepower()


Scraping: https://www.globalfirepower.com/total-population-by-country.php
Scraping: https://www.globalfirepower.com/available-military-manpower.php
Scraping: https://www.globalfirepower.com/manpower-fit-for-military-service.php
Scraping: https://www.globalfirepower.com/manpower-reaching-military-age-annually.php
Scraping: https://www.globalfirepower.com/active-military-manpower.php
Scraping: https://www.globalfirepower.com/active-reserve-military-manpower.php
Scraping: https://www.globalfirepower.com/manpower-paramilitary.php
Scraping: https://www.globalfirepower.com/capital-cities-by-total-population.php
Scraping: https://www.globalfirepower.com/aircraft-total.php
Scraping: https://www.globalfirepower.com/aircraft-total-fighters.php
Scraping: https://www.globalfirepower.com/aircraft-total-attack-types.php
Scraping: https://www.globalfirepower.com/aircraft-total-transports.php
Scraping: https://www.globalfirepower.com/aircraft-total-trainers.php
Scraping: https://www.globalfirepower.co